# Multimodal House Price — Simplified Fusion Notebook

Combines:
- **Tabular model**: sklearn ensemble (RF + LGBM)
- **Image model**: CLIP ViT-B/32 → MLP

Two fusion approaches:
1. **Stage 1** — Prediction blending (simple baseline)
2. **Stage 2** — Feature-level fusion MLP (recommended)

## 0. Setup

In [1]:
import os, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import clip

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor

import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
RANDOM_STATE = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

c:\Users\user\anaconda3\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


Device: cpu


## 1. Load & Align Both Datasets

`house_id` is the key that links tabular data to images.
We merge everything into one DataFrame so splits are always consistent.

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
TABULAR_CSV = Path('../data/satilir_model_base_with_house_id.csv')
IMAGE_CSV   = Path('../data/satilir_id_price_folder.csv')

PHOTO_ROOT_CANDIDATES = [
    Path('../satilir_photos/satilir_photos'),
    Path('../satilir_photos'),
]
PHOTOS_DIR = next((p for p in PHOTO_ROOT_CANDIDATES if p.exists()), PHOTO_ROOT_CANDIDATES[0])

OUTPUT_DIR = Path('../models/multimodal')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXT  = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
MAX_IMAGES = 20
IMG_SIZE   = 224

print('Tabular CSV:', TABULAR_CSV)
print('Image CSV  :', IMAGE_CSV)
print('Photos dir :', PHOTOS_DIR.resolve())

Tabular CSV: ..\data\satilir_model_base_with_house_id.csv
Image CSV  : ..\data\satilir_id_price_folder.csv
Photos dir : E:\Coding\ai_project\satilir_photos\satilir_photos


In [3]:
# ── Load tabular data ────────────────────────────────────────────────────────
df_tab = pd.read_csv(TABULAR_CSV)
df_tab['house_id'] = pd.to_numeric(df_tab['house_id'], errors='coerce').astype('Int64')
df_tab = df_tab.dropna(subset=['house_id']).reset_index(drop=True)
print('Tabular shape:', df_tab.shape)

Tabular shape: (4412, 59)


In [4]:
# ── Load image CSV & get image paths ─────────────────────────────────────────
df_img = pd.read_csv(IMAGE_CSV)
df_img['house_id'] = pd.to_numeric(df_img['house_id'], errors='coerce').astype('Int64')
df_img['price']    = pd.to_numeric(df_img['price'], errors='coerce')
df_img = df_img.dropna(subset=['house_id', 'price', 'image_folder_name']).reset_index(drop=True)

def get_image_paths(folder_name):
    folder = PHOTOS_DIR / str(folder_name)
    if not folder.exists():
        return []
    return [f for f in sorted(folder.iterdir()) if f.suffix.lower() in VALID_EXT]

df_img['img_paths'] = df_img['image_folder_name'].apply(get_image_paths)
df_img['n_images']  = df_img['img_paths'].apply(len)
df_img = df_img[df_img['n_images'] >= 1].reset_index(drop=True)
print('Image CSV shape after filtering:', df_img.shape)

Image CSV shape after filtering: (4412, 8)


In [5]:
# ── Merge on house_id (inner join) ───────────────────────────────────────────
img_cols = ['house_id', 'image_folder_name', 'img_paths', 'n_images']
if 'price' not in df_tab.columns:
    img_cols.append('price')

df = df_tab.merge(df_img[img_cols], on='house_id', how='inner').reset_index(drop=True)
print(f'Merged rows: {len(df):,}  (tabular: {len(df_tab):,}, image: {len(df_img):,})')

Merged rows: 4,412  (tabular: 4,412, image: 4,412)


## 2. Split the Data

Split once by `house_id`. Everything — tabular and image — uses this same split.

In [6]:
PRICE_COL = 'num__price' if 'num__price' in df.columns else 'price'

# Remove extreme prices
df = df[df[PRICE_COL] < 10_000_000].reset_index(drop=True)

# Stratified split on price bins
df['price_bin'] = pd.qcut(df[PRICE_COL], q=5, labels=False, duplicates='drop')
all_ids = df['house_id'].unique()
id_to_bin = df.drop_duplicates('house_id').set_index('house_id')['price_bin']

train_ids, temp_ids = train_test_split(
    all_ids, test_size=0.28, random_state=RANDOM_STATE,
    stratify=id_to_bin.loc[all_ids]
)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=RANDOM_STATE)

train_df = df[df['house_id'].isin(train_ids)].copy().reset_index(drop=True)
val_df   = df[df['house_id'].isin(val_ids)].copy().reset_index(drop=True)
test_df  = df[df['house_id'].isin(test_ids)].copy().reset_index(drop=True)

# Cap outlier prices on train only
price_cap = float(train_df[PRICE_COL].quantile(0.98))
train_df = train_df[train_df[PRICE_COL] <= price_cap].reset_index(drop=True)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

Train: 3,112  Val: 618  Test: 618


## 3. Tabular Model

In [7]:
# ── Define features ──────────────────────────────────────────────────────────
EXCLUDE_COLS = {'house_id', PRICE_COL, 'price_bin', 'image_folder_name',
                'img_paths', 'n_images', 'price'}

TAB_FEATURES = [c for c in train_df.columns if c not in EXCLUDE_COLS
                and train_df[c].dtype in [np.float64, np.float32, np.int64, np.int32, float, int]]

print(f'Tabular feature count: {len(TAB_FEATURES)}')

X_train_tab = train_df[TAB_FEATURES].values
X_val_tab   = val_df[TAB_FEATURES].values
X_test_tab  = test_df[TAB_FEATURES].values

y_train = train_df[PRICE_COL].values.astype(float)
y_val   = val_df[PRICE_COL].values.astype(float)
y_test  = test_df[PRICE_COL].values.astype(float)

Tabular feature count: 54


In [8]:
# ── Train RF + LGBM ensemble ─────────────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=2000, max_depth=None, max_features=0.7,
    min_samples_split=4, min_samples_leaf=1, bootstrap=True,
    random_state=RANDOM_STATE, n_jobs=-1,
)
lgbm = LGBMRegressor(
    n_estimators=1600, learning_rate=0.03, num_leaves=63,
    subsample=0.85, colsample_bytree=0.8, min_child_samples=20,
    reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1,
)

print('Fitting RF...')
rf.fit(X_train_tab, y_train)
print('Fitting LGBM...')
lgbm.fit(X_train_tab, y_train)

# Equal blend
def tab_predict(X):
    return np.clip(0.5 * rf.predict(X) + 0.5 * lgbm.predict(X), 0, None)

tab_pred_train = tab_predict(X_train_tab)
tab_pred_val   = tab_predict(X_val_tab)
tab_pred_test  = tab_predict(X_test_tab)

print(f'Tabular val R2  : {r2_score(y_val, tab_pred_val):.4f}')
print(f'Tabular val MAE : {mean_absolute_error(y_val, tab_pred_val):,.0f}')

Fitting RF...
Fitting LGBM...
Tabular val R2  : 0.7641
Tabular val MAE : 28,320


## 4. Image Model — Extract CLIP Embeddings

Extract once, save to disk. Reused in both Stage 1 and Stage 2.

In [9]:
# ── Load CLIP ────────────────────────────────────────────────────────────────
clip_model, _ = clip.load('ViT-B/32', device=device)
clip_model     = clip_model.float().eval()
CLIP_DIM       = clip_model.visual.output_dim  # 512

for param in clip_model.parameters():
    param.requires_grad = False

tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.48145466, 0.4578275, 0.40821073),
        std =(0.26862954, 0.26130258, 0.27577711),
    ),
])
print(f'CLIP loaded. Each house → {CLIP_DIM}-d embedding.')

CLIP loaded. Each house → 512-d embedding.


In [10]:
# ── Extract & cache embeddings ───────────────────────────────────────────────
EMBED_CACHE = OUTPUT_DIR / 'multimodal_embeddings.pt'

@torch.no_grad()
def embed_house(img_paths):
    tensors = []
    for path in img_paths[:MAX_IMAGES]:
        try:
            tensors.append(tfm(Image.open(path).convert('RGB')))
        except Exception:
            continue
    if not tensors:
        return torch.zeros(CLIP_DIM)
    batch = torch.stack(tensors).to(device)
    embs  = clip_model.encode_image(batch)
    embs  = F.normalize(embs.float(), dim=-1)
    return embs.mean(dim=0).cpu()

if EMBED_CACHE.exists():
    embeddings = torch.load(EMBED_CACHE)
    print(f'Loaded {len(embeddings):,} cached embeddings.')
else:
    print('Extracting embeddings (runs once)...')
    embeddings = {}
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Embedding'):
        hid = int(row['house_id'])
        if hid not in embeddings:
            embeddings[hid] = embed_house(row['img_paths'])
    torch.save(embeddings, EMBED_CACHE)
    print(f'Done. {len(embeddings):,} embeddings saved.')

Loaded 4,412 cached embeddings.


## 5. Shared helpers (log-price normalisation & MdAPE)

In [11]:
# Log-normalise target based on train distribution only
log_prices = np.log1p(y_train)
Y_MEAN = float(log_prices.mean())
Y_STD  = float(log_prices.std()) + 1e-8

def to_price(x_norm):
    return np.expm1(np.clip(x_norm * Y_STD + Y_MEAN, 6.0, 16.5))

def norm_price(price):
    return (np.log1p(price) - Y_MEAN) / Y_STD

def mdape(true, pred):
    return float(np.median(np.abs(true - pred) / (true + 1e-8)) * 100)

## Stage 1 — Prediction Blending

```
Tabular ensemble → pred_tab  ─┐
                               ├→ α·pred_tab + (1-α)·pred_img → final price
CLIP → MLP       → pred_img  ─┘
```

In [12]:
# ── Image-only dataset & loaders ─────────────────────────────────────────────
class ImageDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        emb    = embeddings[int(row['house_id'])]
        target = torch.tensor(norm_price(float(row[PRICE_COL])), dtype=torch.float32)
        return emb, target

BATCH_SIZE = 64
img_train_loader = DataLoader(ImageDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
img_val_loader   = DataLoader(ImageDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False)
img_test_loader  = DataLoader(ImageDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False)

In [13]:
# ── Train image-only MLP ─────────────────────────────────────────────────────
EPOCHS   = 100
LR       = 3e-4
PATIENCE = 10

img_model = nn.Sequential(
    nn.LayerNorm(CLIP_DIM),
    nn.Linear(CLIP_DIM, 512), nn.GELU(), nn.Dropout(0.3),
    nn.Linear(512, 128),      nn.GELU(), nn.Dropout(0.15),
    nn.Linear(128, 1),
).to(device)

criterion = nn.HuberLoss(delta=0.5)
optimizer = torch.optim.AdamW(img_model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(img_train_loader), epochs=EPOCHS
)

@torch.no_grad()
def eval_img_model(model, loader):
    model.eval()
    preds, targets = [], []
    for emb, target in loader:
        p = model(emb.to(device)).squeeze(1)
        preds.append(p.cpu().numpy())
        targets.append(target.numpy())
    return to_price(np.concatenate(preds)), to_price(np.concatenate(targets))

best_mdape, best_weights, no_improve = float('inf'), None, 0
print(f'{"Ep":>4} | {"Loss":>7} | {"Val MAE":>11} | {"MdAPE":>7}')
print('-' * 38)

for epoch in range(1, EPOCHS + 1):
    img_model.train()
    total_loss = 0.0
    for emb, target in img_train_loader:
        emb, target = emb.to(device), target.to(device)
        pred = img_model(emb).squeeze(1)
        loss = criterion(pred, target)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    preds_azn, targets_azn = eval_img_model(img_model, img_val_loader)
    val_mae   = mean_absolute_error(targets_azn, preds_azn)
    val_mdape = mdape(targets_azn, preds_azn)
    marker = '*' if val_mdape < best_mdape else ' '
    if val_mdape < best_mdape:
        best_mdape   = val_mdape
        best_weights = {k: v.clone() for k, v in img_model.state_dict().items()}
        no_improve   = 0
    else:
        no_improve += 1
    print(f'{marker}{epoch:3d}/{EPOCHS} | {total_loss/len(img_train_loader):.4f} | {val_mae:>11,.0f} | {val_mdape:>6.2f}%')
    if no_improve >= PATIENCE:
        print('Early stop'); break

img_model.load_state_dict(best_weights)
print(f'\nBest val MdAPE (image-only): {best_mdape:.2f}%')

  Ep |    Loss |     Val MAE |   MdAPE
--------------------------------------
*  1/100 | 0.2735 |      69,564 |  38.45%
   2/100 | 0.2698 |      69,021 |  38.68%
*  3/100 | 0.2673 |      68,296 |  38.35%
*  4/100 | 0.2625 |      67,394 |  38.20%
*  5/100 | 0.2587 |      66,632 |  37.59%
*  6/100 | 0.2531 |      65,481 |  36.76%
*  7/100 | 0.2503 |      64,656 |  36.49%
*  8/100 | 0.2448 |      63,293 |  35.29%
*  9/100 | 0.2409 |      61,983 |  34.11%
* 10/100 | 0.2379 |      61,297 |  33.71%
* 11/100 | 0.2341 |      60,406 |  32.16%
* 12/100 | 0.2306 |      60,335 |  31.76%
* 13/100 | 0.2288 |      59,193 |  30.79%
* 14/100 | 0.2250 |      58,966 |  30.63%
* 15/100 | 0.2240 |      58,168 |  29.72%
* 16/100 | 0.2210 |      57,382 |  29.68%
  17/100 | 0.2183 |      57,587 |  30.02%
  18/100 | 0.2164 |      57,626 |  30.17%
* 19/100 | 0.2173 |      56,172 |  29.28%
* 20/100 | 0.2153 |      55,503 |  29.28%
* 21/100 | 0.2141 |      55,653 |  29.08%
  22/100 | 0.2138 |      55,153 |  29.35

In [14]:
# ── Find best blend weight on validation, then evaluate on test ───────────────
img_pred_train, _ = eval_img_model(img_model, img_train_loader)
img_pred_val,   _ = eval_img_model(img_model, img_val_loader)
img_pred_test,  _ = eval_img_model(img_model, img_test_loader)

best_alpha, best_val_r2 = 0.5, -1e9
for alpha in np.linspace(0, 1, 21):
    r2 = r2_score(y_val, alpha * tab_pred_val + (1 - alpha) * img_pred_val)
    if r2 > best_val_r2:
        best_val_r2 = r2
        best_alpha  = alpha

blended_test = best_alpha * tab_pred_test + (1 - best_alpha) * img_pred_test

print(f'Best alpha (tabular weight): {best_alpha:.2f}')
print(f'Val  R2 (blended)           : {best_val_r2:.4f}')
print()
print('── Stage 1 TEST RESULTS ──')
print(f'  R2    : {r2_score(y_test, blended_test):.4f}')
print(f'  MAE   : {mean_absolute_error(y_test, blended_test):,.0f} AZN')
print(f'  MdAPE : {mdape(y_test, blended_test):.2f}%')
print()
print(f'  (tabular-only R2) : {r2_score(y_test, tab_pred_test):.4f}')
print(f'  (image-only  R2)  : {r2_score(y_test, img_pred_test):.4f}')

Best alpha (tabular weight): 0.85
Val  R2 (blended)           : 0.7742

── Stage 1 TEST RESULTS ──
  R2    : 0.7407
  MAE   : 30,094 AZN
  MdAPE : 12.63%

  (tabular-only R2) : 0.7341
  (image-only  R2)  : 0.3776


## Stage 2 — Feature-Level Fusion MLP

```
CLIP embedding [512]      ─────────────────────┐
                                                ├→ Concat → MLP → price
Tabular features [N_tab] → Encoder → Norm  ───┘
```

In [15]:
# ── Normalise tabular features for the neural net ─────────────────────────────
scaler = StandardScaler()
X_train_tab_norm = scaler.fit_transform(X_train_tab).astype(np.float32)
X_val_tab_norm   = scaler.transform(X_val_tab).astype(np.float32)
X_test_tab_norm  = scaler.transform(X_test_tab).astype(np.float32)

N_TAB = X_train_tab_norm.shape[1]
print(f'Tabular feature dim : {N_TAB}')
print(f'CLIP embedding dim  : {CLIP_DIM}')

Tabular feature dim : 54
CLIP embedding dim  : 512


In [16]:
# ── Multimodal dataset & loaders ─────────────────────────────────────────────
class MultimodalDataset(Dataset):
    def __init__(self, dataframe, tab_features_norm):
        self.df  = dataframe.reset_index(drop=True)
        self.tab = torch.tensor(tab_features_norm, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        emb    = embeddings[int(row['house_id'])]
        tab    = self.tab[idx]
        target = torch.tensor(norm_price(float(row[PRICE_COL])), dtype=torch.float32)
        return emb, tab, target

mm_train_loader = DataLoader(MultimodalDataset(train_df, X_train_tab_norm), batch_size=BATCH_SIZE, shuffle=True)
mm_val_loader   = DataLoader(MultimodalDataset(val_df,   X_val_tab_norm),   batch_size=BATCH_SIZE, shuffle=False)
mm_test_loader  = DataLoader(MultimodalDataset(test_df,  X_test_tab_norm),  batch_size=BATCH_SIZE, shuffle=False)
print('Multimodal DataLoaders ready.')

Multimodal DataLoaders ready.


In [17]:
# ── Fusion model definition ───────────────────────────────────────────────────
class FusionModel(nn.Module):
    def __init__(self, clip_dim, tab_dim, img_hidden=256, tab_hidden=128, dropout=0.3):
        super().__init__()

        self.img_branch = nn.Sequential(
            nn.LayerNorm(clip_dim),
            nn.Linear(clip_dim, img_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.tab_branch = nn.Sequential(
            nn.BatchNorm1d(tab_dim),
            nn.Linear(tab_dim, tab_hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(tab_hidden * 2, tab_hidden),
            nn.GELU(),
        )
        fused_dim = img_hidden + tab_hidden
        self.head = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 64),        nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(self, img_emb, tab_feat):
        return self.head(torch.cat([self.img_branch(img_emb),
                                    self.tab_branch(tab_feat)], dim=1)).squeeze(1)

mm_model = FusionModel(CLIP_DIM, N_TAB).to(device)
print(f'Fusion model params: {sum(p.numel() for p in mm_model.parameters()):,}')

Fusion model params: 295,277


In [18]:
# ── Train fusion model ───────────────────────────────────────────────────────
MM_EPOCHS   = 150
MM_LR       = 3e-4
MM_PATIENCE = 15

mm_criterion = nn.HuberLoss(delta=0.5)
mm_optimizer = torch.optim.AdamW(mm_model.parameters(), lr=MM_LR, weight_decay=1e-2)
mm_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    mm_optimizer, max_lr=MM_LR,
    steps_per_epoch=len(mm_train_loader), epochs=MM_EPOCHS
)

@torch.no_grad()
def eval_mm_model(model, loader):
    model.eval()
    preds, targets = [], []
    for emb, tab, target in loader:
        p = model(emb.to(device), tab.to(device))
        preds.append(p.cpu().numpy())
        targets.append(target.numpy())
    return to_price(np.concatenate(preds)), to_price(np.concatenate(targets))

mm_best_mdape, mm_best_weights, mm_no_improve = float('inf'), None, 0
history = {'loss': [], 'val_mdape': [], 'val_mae': []}

print(f'{"Ep":>4} | {"Loss":>7} | {"Val MAE":>11} | {"MdAPE":>7}')
print('-' * 38)

for epoch in range(1, MM_EPOCHS + 1):
    mm_model.train()
    total_loss = 0.0
    for emb, tab, target in mm_train_loader:
        emb, tab, target = emb.to(device), tab.to(device), target.to(device)
        pred = mm_model(emb, tab)
        loss = mm_criterion(pred, target)
        mm_optimizer.zero_grad(); loss.backward(); mm_optimizer.step()
        total_loss += loss.item()
    mm_scheduler.step()

    preds_azn, targets_azn = eval_mm_model(mm_model, mm_val_loader)
    val_mae   = mean_absolute_error(targets_azn, preds_azn)
    val_mdape = mdape(targets_azn, preds_azn)

    history['loss'].append(total_loss / len(mm_train_loader))
    history['val_mae'].append(val_mae)
    history['val_mdape'].append(val_mdape)

    marker = '*' if val_mdape < mm_best_mdape else ' '
    if val_mdape < mm_best_mdape:
        mm_best_mdape   = val_mdape
        mm_best_weights = {k: v.clone() for k, v in mm_model.state_dict().items()}
        mm_no_improve   = 0
    else:
        mm_no_improve += 1
    print(f'{marker}{epoch:3d}/{MM_EPOCHS} | {total_loss/len(mm_train_loader):.4f} | {val_mae:>11,.0f} | {val_mdape:>6.2f}%')
    if mm_no_improve >= MM_PATIENCE:
        print('Early stop'); break

mm_model.load_state_dict(mm_best_weights)
print(f'\nBest val MdAPE (fusion): {mm_best_mdape:.2f}%')

  Ep |    Loss |     Val MAE |   MdAPE
--------------------------------------
*  1/150 | 0.2724 |      68,468 |  37.93%
*  2/150 | 0.2640 |      66,252 |  36.67%
*  3/150 | 0.2565 |      62,042 |  33.77%
*  4/150 | 0.2405 |      55,367 |  29.39%
*  5/150 | 0.2161 |      44,556 |  23.45%
*  6/150 | 0.1908 |      41,857 |  21.34%
*  7/150 | 0.1775 |      41,993 |  21.30%
   8/150 | 0.1751 |      41,868 |  21.54%
*  9/150 | 0.1734 |      40,677 |  20.78%
  10/150 | 0.1673 |      40,899 |  20.84%
* 11/150 | 0.1672 |      38,989 |  19.92%
* 12/150 | 0.1643 |      39,117 |  19.56%
  13/150 | 0.1594 |      40,969 |  20.72%
  14/150 | 0.1577 |      39,379 |  20.46%
  15/150 | 0.1579 |      38,533 |  20.12%
  16/150 | 0.1569 |      39,300 |  20.43%
  17/150 | 0.1544 |      38,889 |  20.04%
  18/150 | 0.1537 |      38,601 |  20.57%
  19/150 | 0.1522 |      39,766 |  20.78%
  20/150 | 0.1513 |      38,360 |  20.63%
  21/150 | 0.1481 |      39,088 |  21.23%
  22/150 | 0.1485 |      39,253 |  21.17

## 6. Final Results

In [ ]:
mm_preds_test, mm_targets_test = eval_mm_model(mm_model, mm_test_loader)

results = {
    'Tabular only (RF+LGBM)' : (r2_score(y_test, tab_pred_test),
                                 mean_absolute_error(y_test, tab_pred_test),
                                 mdape(y_test, tab_pred_test)),
    'Image only (CLIP+MLP)'  : (r2_score(y_test, img_pred_test),
                                 mean_absolute_error(y_test, img_pred_test),
                                 mdape(y_test, img_pred_test)),
    'Stage 1: Blend'         : (r2_score(y_test, blended_test),
                                 mean_absolute_error(y_test, blended_test),
                                 mdape(y_test, blended_test)),
    'Stage 2: Fusion MLP'    : (r2_score(mm_targets_test, mm_preds_test),
                                 mean_absolute_error(mm_targets_test, mm_preds_test),
                                 mdape(mm_targets_test, mm_preds_test)),
}

print(f'\n{"Model":<28} {"R2":>7} {"MAE":>12} {"MdAPE":>8}')
print('-' * 60)
for name, (r2, mae, md) in results.items():
    print(f'{name:<28} {r2:>7.4f} {mae:>12,.0f} {md:>7.2f}%')

In [ ]:
# ── Training curves + scatter plot ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

eps = range(1, len(history['loss']) + 1)
axes[0].plot(eps, history['loss'])
axes[0].set_title('Fusion — Train Loss'); axes[0].grid(alpha=0.3)

axes[1].plot(eps, history['val_mdape'], color='purple')
axes[1].set_title('Fusion — Val MdAPE %'); axes[1].grid(alpha=0.3)

cap  = np.percentile(mm_targets_test, 95)
mask = mm_targets_test < cap
axes[2].scatter(mm_targets_test[mask]/1000, mm_preds_test[mask]/1000, alpha=0.3, s=6)
mn, mx = mm_targets_test[mask].min()/1000, mm_targets_test[mask].max()/1000
axes[2].plot([mn, mx], [mn, mx], 'r--', label='perfect')
axes[2].set_title(f'Fusion: Pred vs Actual  R2={r2_score(mm_targets_test, mm_preds_test):.3f}')
axes[2].set_xlabel('Actual (K AZN)'); axes[2].set_ylabel('Predicted (K AZN)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'multimodal_results.png', dpi=120)
plt.show()

## 7. Save Models

In [ ]:
torch.save({
    'fusion_model_state' : mm_model.state_dict(),
    'img_model_state'    : img_model.state_dict(),
    'scaler_mean'        : scaler.mean_,
    'scaler_scale'       : scaler.scale_,
    'tab_features'       : TAB_FEATURES,
    'y_mean'             : Y_MEAN,
    'y_std'              : Y_STD,
    'blend_alpha'        : best_alpha,
    'clip_dim'           : CLIP_DIM,
    'n_tab'              : N_TAB,
}, OUTPUT_DIR / 'multimodal_model.pt')
print('Models saved to', OUTPUT_DIR)

## 8. Predict for a New House

In [ ]:
def predict_multimodal(row_df):
    """
    row_df: single-row DataFrame with tabular features + 'house_id' + 'img_paths'
    """
    hid = int(row_df['house_id'].values[0])
    emb = embeddings[hid] if hid in embeddings else embed_house(row_df['img_paths'].values[0])

    tab_raw  = row_df[TAB_FEATURES].values.astype(np.float32)
    tab_norm = scaler.transform(tab_raw)

    mm_model.eval()
    with torch.no_grad():
        pred_norm = mm_model(emb.unsqueeze(0).to(device),
                             torch.tensor(tab_norm, dtype=torch.float32).to(device)).item()

    price = float(to_price(np.array([pred_norm]))[0])
    print(f'Fusion model estimate: {price:,.0f} AZN')
    return price

# Demo on first test house
demo_row = test_df.iloc[[0]]
print(f'Actual price: {demo_row[PRICE_COL].values[0]:,.0f} AZN')
predict_multimodal(demo_row)